# IDAC Data Preprocessing

This notebook performs data preprocessing operations on the IDAC Excel dataset. You should run this prior to working with data in Tableau if you need any data preprocessing.

Behavior is controlled via environment variables specified in a .env file placed outside this directory. 
A sample .env file is included - you'll need to modify it prior to running this notebook.

In [ ]:
from dotenv import load_dotenv

loaded_vars = load_dotenv(verbose=True)
if not loaded_vars:
    raise Exception('Unable to load any environment variables - make sure you have a .env file')


## Load Dataset

Load Excel dataset into memory. All sheets will be read, so this operation may take awhile.

Requires `IDAC_DATASET_PATH` environment variable to be set, like in .env file.

Ex: `IDAC_DATASET_PATH = folder1/folder2/filename.xlsx` will load the Excel file at "folder1/folder2/filename.xlsx".

In [ ]:
import pandas as pd
import os

idac_dataset = pd.read_excel(os.getenv('IDAC_DATASET_PATH'), sheet_name=None)

idac_dataset.keys()

## Operation 1: Remove Irrelevant Columns

Certain columns in one sheet are irrelevant in another sheet. For example, LEV Simple Code is only relevant to cases disposed but is included in cases filed as well.

This operation removes unnecessary/irrelevant columns from sheets. It consists of sub operations applied to specific sheets, so you can control which sheets the general removal operation is performed on.

Requires `DO_OP1_APPEALS`, `DO_OP1_JUVENILE`, `DO_OP1_ADULT_FILED`, `DO_OP1_ADULT_DISPO` environment variables to be set to either "y" or "n". If "y", the operation will be performed on the sheet. If "n", the operation will not be performed on the sheet.

Ex: `DO_OP1_ADULT_FILED = y` will perform irrelevant column removal on Adult Cases Filed sheet in Excel file.

In [ ]:
import pandas as pd
import os

if os.getenv('DO_OP1_APPEALS').lower() == 'y':
    drop_cols = [
        'LEV Simple Code',
        'No Attorney Present',
        'Waived',
        'Unknown',
        'N/A: No Formal Hearing Held',
        'Homicide',
        'CSPCaseType',
        'Case Type Recode',
        'OffenseGrade',
        'Status',
        ]
    
    print('Performing irrelevant column removal on Appeals sheet')
    print(f'The following columns will be removed: {drop_cols}')

    appeals_df = idac_dataset['Appeals']
    appeals_df = appeals_df.drop(columns=drop_cols)
    idac_dataset['Appeals'] = appeals_df

    print(f'Remaining columns in Appeals sheet: {appeals_df.columns}')
    

In [ ]:
import pandas as pd
import os

if os.getenv('DO_OP1_JUVENILE').lower() == 'y':
    drop_cols = [
        'Other',
        'No indication of defendant representation type',
        'Pro Se',
        'Homicide',
        'Status',
        'CaseCategory',
    ]

    print('Performing irrelevant column removal on Juvenile Cases Filed & Dispo sheet')
    print(f'The following columns will be removed: {drop_cols}')

    juvenile_df = idac_dataset['Juvenile Cases Filed & Dispo']
    juvenile_df = juvenile_df.drop(columns=drop_cols)
    idac_dataset['Juvenile Cases Filed & Dispo'] = juvenile_df

    print(f'Remaining columns in Juvenile Cases Filed & Dispo sheet: {juvenile_df.columns}')


In [ ]:
import pandas as pd
import os

if os.getenv('DO_OP1_ADULT_FILED').lower() == 'y':
    drop_cols = [
        'LEV Simple Code',
        'No Attorney Present',
        'Waived',
        'Unknown',
        'N/A: No Formal Hearing Held',
        'Pro Se',
        'Post Disposition Attorney',
        'CaseCategory',
    ]

    print('Performing irrelevant column removal on Adults Cases Filed sheet')
    print(f'The following columns will be removed: {drop_cols}')

    adult_filed = idac_dataset['Adult Cases Filed']
    adult_filed = adult_filed.drop(columns=drop_cols)
    idac_dataset['Adult Cases Filed'] = adult_filed

    print(f'Remaining columns in Adult Cases Filed sheet: {adult_filed.columns}')


In [ ]:
import pandas as pd
import os

if os.getenv('DO_OP1_ADULT_DISPO').lower() == 'y':
    drop_cols = [
        'No Attorney Present',
        'Waived',
        'Unknown',
        'N/A: No Formal Hearing Held',
        'Pro Se',
        'CaseCategory',
        'Secondary Disposition Code',
    ]

    print('Performing irrelevant column removal on Adults Cases Dispo sheet')
    print(f'The following columns will be removed: {drop_cols}')

    adult_dispo = idac_dataset['Adult Cases Dispo']
    adult_dispo = adult_dispo.drop(columns=drop_cols)
    idac_dataset['Adult Cases Dispo'] = adult_dispo

    print(f'Remaining columns in Adult Cases Dispo sheet: {adult_dispo.columns}')


## Operation 2: Split Juvenile Cases Filed & Dispo Sheet

Currently, the Juvenile Cases Filed & Dispo sheet contains juveniles cases filed AND juvenile cases disposed. This is unlike adult cases, which have separate sheets for the cases filed metric and cases disposed metric.

This operation "splits" the Juvenile Cases Filed & Dispo sheet into 2 separate sheets: 1 for just the cases filed metric (Juvenile Cases Filed), and the other for the cases disposed metric (Juvenile Cases Dispo).

**NOTE: If `DO_OP1_JUVENILE` was set to "y", this operation will also remove irrelevant columns that may not have been previously removed.**

Requires `DO_OP2_JUVENILE` environment variable to be set to either "y" or "n". If "y", the operation will be performed on the sheet. If "n", the operation will not be performed on the sheet.

Ex: `DO_OP2_JUVENILE = y` will result in the output Excel file containing 2 sheets called Juvenile Cases Filed and Juvenile Cases Dispo, each containing data regarding their corresponding metric.

In [ ]:
import pandas as pd
import os
from IPython.display import display

if os.getenv('DO_OP2_JUVENILE').lower() == 'y' and 'Juvenile Cases Filed & Dispo' in idac_dataset:
    juvenile_df = idac_dataset['Juvenile Cases Filed & Dispo']

    juvenile_filed = juvenile_df.loc[juvenile_df['Metric Type'] == 'Cases Filed']
    juvenile_dispo = juvenile_df.loc[juvenile_df['Metric Type'] == 'Cases Disposed']

    if os.getenv('DO_OP1_JUVENILE').lower() == 'y':
        # additionally remove irrelevant columns
        drop_cols_filed = [
            'LEV Simple Code',
            'Post Disposition Attorney',  
        ]
        print(f'Removing {drop_cols_filed} columns from Juvenile Cases Filed sheet')
        juvenile_filed = juvenile_filed.drop(columns=drop_cols_filed)
        print(f'Remaining columns in Juvenile Cases Filed sheet: {juvenile_filed.columns}')

        drop_cols_dispo = [
            'OTN with more than 1 AttorneyRepresentationType',
        ]
        print(f'Removing {drop_cols_dispo} columns from Juvenile Dispo Filed sheet')
        juvenile_dispo = juvenile_dispo.drop(columns=drop_cols_dispo)
        print(f'Remaining columns in Juvenile Cases Dispo sheet: {juvenile_dispo.columns}')

    display(juvenile_filed.head(n=2))
    display(juvenile_dispo.head(n=2))

    idac_dataset['Juvenile Cases Filed'] = juvenile_filed
    idac_dataset['Juvenile Cases Dispo'] = juvenile_dispo
    del idac_dataset['Juvenile Cases Filed & Dispo']

    display(idac_dataset.keys())


## Operation 3: Convert to Long Format

The raw IDAC dataset is wide. Computers work better/more efficiently with long data.

This operation transforms each sheet to be in long format. I**t assumes the value columns are those of the counts for each sheet**, e.g. Public Defender, Court Appointed, Private, etc. **The other assumption is that the (row) identifier columns are ALL the non-value columns**, e.g. County, Year, Data Date, etc. Naturally, this is sheet-specific. Two new columns are created: Count Type, which stores the type of count variable (e.g. Public Defender), and Count, which stores the actual numeric count of the count variable for that row (e.g. 101).

Requires `DO_OP3_APPEALS`, `DO_OP3_JUVENILE`, `DO_OP3_ADULT_FILED`, `DO_OP3_ADULT_DISPO` environment variables to be set to either "y" or "n". If "y", the operation will be performed on the sheet. If "n", the operation will not be performed on the sheet.

Ex: `DO_OP3_ADULT_FILED = y` will perform wide-to-long format conversion on Adult Cases Filed sheet.

**NOTE: If operation 2 was performed and `DO_OP3_JUVENILE is "y", then long format conversion will be performed on the 2 new sheets resulting from the splitting performed previosuly.**

In [ ]:
import pandas as pd
from IPython.display import display

if os.getenv('DO_OP1_ADULT_FILED').lower() == 'y' and 'Adult Cases Filed' in idac_dataset:
    adult_filed = idac_dataset['Adult Cases Filed']

    val_cols = [
        'Count of cases',
        'Public Defender',
        'Court Appointed',
        'Private',
        'Other',
        'No indication of defendant representation type',
        'Involving indigent defender',
        ]
    id_cols = [col for col in adult_filed.columns.to_list() if col not in val_cols]

    var_name = 'Count Type'
    val_name = 'Count'

    print(f'Identifier columns: {id_cols}')
    print(f'Value columns: {val_cols}')

    # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    new_adult_filed = pd.melt(
        adult_filed,
        id_vars=id_cols,
        value_vars=val_cols,
        var_name=var_name,
        value_name=val_name
    )

    display(new_adult_filed.head(n=5))

    idac_dataset['Adult Cases Filed Long'] = new_adult_filed
    del idac_dataset['Adult Cases Filed']


In [ ]:
import pandas as pd
from IPython.display import display

if os.getenv('DO_OP3_ADULT_DISPO').lower() == 'y' and 'Adult Cases Dispo' in idac_dataset:
    adult_dispo = idac_dataset['Adult Cases Dispo']

    val_cols = [
        'Count of cases',
        'Public Defender',
        'Court Appointed',
        'Private',
        'Other',
        'No indication of defendant representation type',
        'Involving indigent defender',
        ]
    id_cols = [col for col in adult_dispo.columns.to_list() if col not in val_cols]

    var_name = 'Count Type'
    val_name = 'Count'

    print(f'Identifier columns: {id_cols}')
    print(f'Value columns: {val_cols}')

    # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    new_adult_dispo = pd.melt(
        adult_dispo,
        id_vars=id_cols,
        value_vars=val_cols,
        var_name=var_name,
        value_name=val_name
    )

    display(new_adult_dispo.head(n=5))

    idac_dataset['Adult Cases Dispo Long'] = new_adult_dispo
    del idac_dataset['Adult Cases Dispo']


In [ ]:
import pandas as pd
from IPython.display import display

if os.getenv('DO_OP3_APPEALS').lower() == 'y' and 'Appeals' in idac_dataset:
    appeals = idac_dataset['Appeals']

    val_cols = [
        'Count of cases',
        'Public Defender',
        'Court Appointed',
        'Private',
        'Other',
        'No indication of defendant representation type',
        'Involving indigent defender',
        'Pro Se',
        ]
    id_cols = [col for col in appeals.columns.to_list() if col not in val_cols]

    var_name = 'Count Type'
    val_name = 'Count'

    print(f'Identifier columns: {id_cols}')
    print(f'Value columns: {val_cols}')

    # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    new_appeals = pd.melt(
        appeals,
        id_vars=id_cols,
        value_vars=val_cols,
        var_name=var_name,
        value_name=val_name
    )

    display(new_appeals.head(n=5))

    idac_dataset['Appeals Long'] = new_appeals
    del idac_dataset['Appeals']


In [ ]:
import pandas as pd
from IPython.display import display

if os.getenv('DO_OP3_JUVENILE').lower() == 'y':
    # these are the same for any juvenile sheet (dispo/filed)
    val_cols = [
        'Count of cases',
        'Public Defender',
        'Court Appointed',
        'Private',
        'No Attorney Present',
        'Waived',
        'Unknown',
        'N/A: No Formal Hearing Held',
        'Involving indigent defender',
        ]
    
    # if operation 2 was not done prior on juvenile sheet
    if 'Juvenile Cases Filed & Dispo' in idac_dataset:
        juvenile = idac_dataset['Juvenile Cases Filed & Dispo']

        id_cols = [col for col in juvenile.columns.to_list() if col not in val_cols]

        var_name = 'Count Type'
        val_name = 'Count'

        print(f'Identifier columns: {id_cols}')
        print(f'Value columns: {val_cols}')

        # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
        new_juvenile = pd.melt(
            juvenile,
            id_vars=id_cols,
            value_vars=val_cols,
            var_name=var_name,
            value_name=val_name
        )

        display(new_juvenile.head(n=5))

        idac_dataset['Juvenile Cases Filed & Dispo Long'] = new_juvenile
        del idac_dataset['Juvenile Cases Filed & Dispo']

    elif 'Juvenile Cases Filed' in idac_dataset and 'Juvenile Cases Dispo' in idac_dataset:
        # operation 2 was performed - juvenile was split into 2 separate sheets
        juvenile_filed = idac_dataset['Juvenile Cases Filed']
        juvenile_dispo = idac_dataset['Juvenile Cases Dispo']
        
        id_cols_filed = [col for col in juvenile_filed.columns.to_list() if col not in val_cols]
        id_cols_dispo = [col for col in juvenile_dispo.columns.to_list() if col not in val_cols]

        var_name = 'Count Type'
        val_name = 'Count'

        print(f'Identifier columns (filed): {id_cols_filed}')
        print(f'Identifier columns (dispo): {id_cols_dispo}')
        print(f'Value columns (both): {val_cols}')

        # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
        new_juvenile_filed = pd.melt(
            juvenile_filed,
            id_vars=id_cols_filed,
            value_vars=val_cols,
            var_name=var_name,
            value_name=val_name
        )

        new_juvenile_dispo = pd.melt(
            juvenile_dispo,
            id_vars=id_cols_dispo,
            value_vars=val_cols,
            var_name=var_name,
            value_name=val_name
        )

        display(new_juvenile_filed.head(n=5))
        display(new_juvenile_dispo.head(n=5))

        idac_dataset['Juvenile Cases Filed Long'] = new_juvenile_filed
        idac_dataset['Juvenile Cases Dispo Long'] = new_juvenile_dispo
        del idac_dataset['Juvenile Cases Filed']
        del idac_dataset['Juvenile Cases Dispo']


## Save to Disk

With all data preprocessing operations complete, last step is to save modified Excel file to disk. A new file will be created - the old one will not be overwritten.

In [ ]:
import time
import pandas as pd

file_start = time.time()
with pd.ExcelWriter('IDAC Dataset Long.xlsx', engine='xlsxwriter') as xl_writer:
    for sheet_name, df in idac_dataset.items():
        sheet_start = time.time()
        print(f'Writing sheet {sheet_name} to output file...')
        df.to_excel(xl_writer, sheet_name=sheet_name, engine='xlsxwriter', index=False)
        print(f'Wrote sheet {sheet_name} to file in {time.time()-sheet_start} seconds')

print('Saved new file as "IDAC Dataset Long.xlsx"')
print(f'Saved to file in {time.time()-file_start} seconds')